In [55]:
# ============================================================
# Cell 1) Imports + Paths (Windows-safe, project-relative)
#   Expected structure:
#     demographics/
#       code/   (notebook here)
#       data/
#         diagram_csv.csv
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

ROOT = Path.cwd().parents[0]   # if notebook is in demographics/code/
DATA = ROOT / "data"

CSV_PATH = DATA / "diagram_csv.csv"

print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH)
print("exists:", CSV_PATH.exists())

ROOT: c:\Users\junse\Documents\research\demographics\diagram1_overall
CSV_PATH: c:\Users\junse\Documents\research\demographics\diagram1_overall\data\diagram_csv.csv
exists: True


In [56]:
# ============================================================
# Cell 2) Load CSV + Filter (crop 4 only, timepoint t2 if present)
#   Update: remove pathogen_score/cat5, add total viral load only
# ============================================================

df = pd.read_csv(CSV_PATH)
print("Loaded shape:", df.shape)
print("Columns (head):", df.columns.tolist()[:25])

# -----------------------------
# Total viral load (virus-only summary for heatmap)
#   - sum across available virus indicators
#   - log1p transform to reduce extreme skew
# -----------------------------
VIRUS_COLS = [
    "CBPVnorm_copy_per_bee",
    "ABPVnorm_copy_per_bee",
    "KBVnorm_copy_per_bee",
    "VDVnorm_copy_per_bee",
    "SBVnorm_copy_per_bee",
    "BQCVnorm_copy_per_bee",
    "IAPVnorm_copy_per_bee",
    "LSVnorm_copy_per_bee",
]

virus_use = [c for c in VIRUS_COLS if c in df.columns]
if len(virus_use) == 0:
    print("WARNING: No virus columns found in CSV (VIRUS_COLS missing).")
    df["total_viral_load"] = 0.0
    df["total_viral_load_log1p"] = 0.0
else:
    Xv = df[virus_use].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    df["total_viral_load"] = Xv.sum(axis=1)
    df["total_viral_load_log1p"] = np.log1p(df["total_viral_load"])

print("Virus cols used:", virus_use)
print("Has total_viral_load?", "total_viral_load" in df.columns)

# ---- Filter to the 4 crops you want
CROPS_KEEP = {"HBB", "CRA", "CAC", "CAS"}
if "crop" not in df.columns:
    raise ValueError("Missing required column: crop")

df = df[df["crop"].isin(CROPS_KEEP)].copy()
print("After crop filter:", df.shape)
print("Crop counts:\n", df["crop"].value_counts())

# ---- If timepoint column exists, keep only t2
# (If you removed timepoint earlier, this does nothing.)
if "timepoint" in df.columns:
    df = df[df["timepoint"].astype(str).str.lower().eq("t2")].copy()
    print("After timepoint==t2 filter:", df.shape)
else:
    print("No 'timepoint' column found -> assuming already fixed to t2.")

# ---- Year counts (do NOT assume 120; check)
if "year" not in df.columns:
    raise ValueError("Missing required column: year")

print("Counts by year:\n", df["year"].value_counts().sort_index())


Loaded shape: (240, 46)
Columns (head): ['sample_id', 'crop', 'tissue', 'location', 'year', 'label_crop_tissue', 'split', 'province', 'location_clean', 'month', 'exposure', 'exposure_bin', 'plate', 'replicate', 'latitude', 'longitude', 'geo_cluster_100km', 'geo_cluster_100km_lat_center', 'geo_cluster_100km_lon_center', 'pest_bees_sum', 'pest_bees_count_detected', 'pest_bees_max', 'pest_nectar_sum', 'pest_nectar_count_detected', 'pest_nectar_max']
Virus cols used: ['CBPVnorm_copy_per_bee', 'ABPVnorm_copy_per_bee', 'KBVnorm_copy_per_bee', 'VDVnorm_copy_per_bee', 'SBVnorm_copy_per_bee', 'BQCVnorm_copy_per_bee', 'IAPVnorm_copy_per_bee', 'LSVnorm_copy_per_bee']
Has total_viral_load? True
After crop filter: (240, 48)
Crop counts:
 crop
HBB    60
CRA    60
CAC    60
CAS    60
Name: count, dtype: int64
No 'timepoint' column found -> assuming already fixed to t2.
Counts by year:
 year
2020    120
2021    120
Name: count, dtype: int64


In [57]:
# ============================================================
# Cell 3) Helpers + Plot Config
# ============================================================

def encode_categories(series):
    """Encode categories into stable integer codes for color tiles."""
    s = pd.Series(series).astype("category")
    cats = s.cat.categories.tolist()
    mapping = {c: i for i, c in enumerate(cats)}
    codes = pd.Series(series).map(mapping).astype("Int64")
    return codes, mapping

def minmax01(x):
    """Min-max normalize to [0, 1] within the plotted subset."""
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if hi - lo == 0:
        return x * 0
    return (x - lo) / (hi - lo)

# Heat rows (continuous tiles)

# Sorting to make blocks visually grouped
SORT_KEYS = ["crop", "tissue", "geo_cluster_100km", "exposure", "plate", "replicate"]

# Plot at most 120 samples per year (like the reference figure)
MAX_N = 120


In [58]:
# ============================================================
# Cell 1) Imports + Paths + Main Config (CLEANED / FIXED)
#   - Removed duplicate BIN_LABELS definition
#   - Added missing constants used in later cells (TITLE_FS, FONT_FAMILY)
#   - Standardized naming: TILE / ANN_ROW_GAP / HEAT_ROW_GAP / TOP_PAD / etc.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path
from PIL import Image

# -----------------------------
# Text style (used in Cell 4)
# -----------------------------
FONT_FAMILY = "DejaVu Sans"
TITLE_FS = 18

# -----------------------------
# Pest burden label bins (value-based, interpretable)
# (used in Cell 3 make_global_colors)
# -----------------------------
PLABELS = ["≤ 1", "1–10", "10–100", "100–1k", "1k–5k", "> 5k"]

# -----------------------------
# Paths
# -----------------------------
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
CSV_PATH = DATA / "diagram_csv.csv"

OUTDIR = ROOT / "figures" / "crop"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_2020 = OUTDIR / "diagram_2020.pdf"
OUT_2021 = OUTDIR / "diagram_2021.pdf"

# --------------------------
# Crop order
# -----------------------------
CROPS_ORDER = ["HBB", "CRA", "CAC", "CAS"]
CROPS_KEEP  = set(CROPS_ORDER)

# -----------------------------
# Max samples shown
# -----------------------------
MAX_N = 120

# -----------------------------
# Layout constants (used by Cell 4)
# -----------------------------
GAP_CROP = 3.0

TILE = 1.0                 # base tile size
left_pad  = 10.5
right_pad = 14.0           # right margin / space for legends
TOP_PAD   = 2.4

ANN_ROW_GAP     = 0.28
HEAT_ROW_GAP    = 0.22
GAP_ANN_TO_HEAT = 1.0


# Uppercase aliases (some versions of Cell 4 use these)
LEFT_PAD  = left_pad
RIGHT_PAD = right_pad
TOP_PAD   = TOP_PAD   # already uppercase, but keep for clarity

ANN_ROW_GAP     = ANN_ROW_GAP
HEAT_ROW_GAP    = HEAT_ROW_GAP
GAP_ANN_TO_HEAT = GAP_ANN_TO_HEAT

# Also alias tile size names (some code uses TILE, some uses tile)
TILE = TILE
tile = TILE
# Backward-compatible aliases (if some cells still use old names)
tile = TILE
top_pad = TOP_PAD
ann_row_gap = ANN_ROW_GAP
heat_row_gap = HEAT_ROW_GAP
gap_ann_to_heat = GAP_ANN_TO_HEAT

# -----------------------------
# Missing values: draw nothing (true blank)
# -----------------------------
DRAW_NA_TILES = False

# -----------------------------
# Bin scheme for numeric-to-multiclass rows (EDIT IF YOU WANT)
# Example: ≤0, 1–5, 5–10, 11–25, 25–50, ≥51
# -----------------------------
BIN_EDGES  = [-np.inf, 0, 5, 10, 25, 50, np.inf]
BIN_LABELS = ["≤ 0", "1–5", "5–10", "11–25", "25–50", "≥ 51"]

# -----------------------------
# Which numeric columns should be binned and shown as multiclass rows?
# -----------------------------
BINNED_NUM_COLS = [
    "boscalid_bee",
    # "boscalid_nectar",
    # "chlorpyrifos_bee",
]


In [59]:
# 원하는 라벨 매핑 (cluster id -> region short label)
CLUSTER_REGION_SHORT = {
    0: "BC",
    4: "AB (N)",
    3: "AB (S)",
    2: "MB",
    1: "QC",
}

# ---- Geo label (5 groups) ----
if "geo_cluster_100km" in df.columns:
    df["geo_cluster_100km_label"] = df["geo_cluster_100km"].apply(
        lambda x: "BLANK" if pd.isna(x) else CLUSTER_REGION_SHORT.get(int(float(x)), f"G{int(float(x))}")
    )
else:
    df["geo_cluster_100km_label"] = "BLANK"


# ---- Convert numeric columns into binned label columns ----
# We create new columns like "boscalid_bee__bin"
def make_binned_label_col(d, src_col, new_col, edges, labels):
    x = pd.to_numeric(d[src_col], errors="coerce")
    # pd.cut makes categories; we convert to strings; NaN becomes BLANK
    cat = pd.cut(x, bins=edges, labels=labels, include_lowest=True, right=True)
    d[new_col] = cat.astype("object").where(cat.notna(), "BLANK")
    return d



# ---- Numeric-binned multiclass rows (user-defined list BINNED_NUM_COLS) ----
BINNED_ROWS = []
for col in BINNED_NUM_COLS:
    if col in df.columns:
        new_col = f"{col}__bin"
        df = make_binned_label_col(df, col, new_col, BIN_EDGES, BIN_LABELS)
        # Row label shown on left + legend group title uses col name
        BINNED_ROWS.append((new_col, col, True))  # (column, label, show_left_label)

# ---- Annotation rows (categorical) ----

ANNOT_ROWS = [
    ("tissue", "Tissue", True),
    ("geo_cluster_100km_label", "Geo (100km cluster)", True),
]
ANNOT_ROWS = ANNOT_ROWS + BINNED_ROWS


# ---- Heat rows (continuous gradient) ----
HEAT_ROWS = [
    ("total_viral_load_log1p", "Total viral load (log1p)"),
    ("pest_sum_all", "Pest burden (continuous)"),
    ("pest_bees_sum", "Bees sum"),
    ("pest_nectar_sum", "Nectar sum"),
    ("pest_pollen_sum", "Pollen sum"),
    # ("pest_wax_sum", "Wax sum"),  # all 0 -> drop

    ("pest_bees_count_detected", "Bees # detected"),
    ("pest_nectar_count_detected", "Nectar # detected"),
    ("pest_pollen_count_detected", "Pollen # detected"),

    ("pest_sum_all", "Total sum"),
]


# ---- Sorting ----
SORT_KEYS = ["crop", "tissue", "geo_cluster_100km_label", "plate", "replicate"]


In [60]:
# ============================================================
# Cell 3) Color Maps + Legend Helpers (SOFT / PAPER-STYLE) — slightly richer
#   Changes vs your version:
#   - Less whitening: SOFT_ALPHA 0.35 -> 0.18
#   - Use deeper colormap bands for pest + binned rows
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def blend_with_white(rgba, alpha=0.18):
    """
    Blend an RGBA color toward white.
    alpha=0   -> original color
    alpha=1.0 -> pure white
    """
    if rgba is None:
        return None
    r, g, b, a = rgba
    r2 = r + (1 - r) * alpha
    g2 = g + (1 - g) * alpha
    b2 = b + (1 - b) * alpha
    return (r2, g2, b2, a)

def minmax01(x):
    x = pd.to_numeric(x, errors="coerce")
    if x.notna().sum() == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if hi - lo == 0:
        return x * 0
    return (x - lo) / (hi - lo)

def draw_grouped_legend_autocol(ax, x_left, y_top, y_bottom, groups,
                                col_width=6.2, col_gap=1.2,
                                title_fs=12, item_fs=11,
                                line_gap=0.62, box=0.30, indent=0.10,
                                pad_title=0.20, pad_group=0.50):
    """
    Draw grouped legends. If we run out of vertical space, continue in a new column to the right.
    Supports:
      - color tuple -> filled square
      - "BLANK_BOX" -> outlined square (meaning true blank / missing)
    """
    x = x_left
    y = y_top

    def draw_blank_box(x0, y0):
        ax.add_patch(Rectangle((x0, y0 - box + 0.06), box, box,
                               facecolor="none", edgecolor="black", linewidth=1.0))

    for group_title, items in groups:
        if y - (pad_title + line_gap) < y_bottom:
            x += col_width + col_gap
            y = y_top

        ax.text(x + col_width / 2.0, y, group_title, ha="center", va="top", fontsize=title_fs)
        y -= (pad_title + line_gap)

        for label, color in items:
            if y - line_gap < y_bottom:
                x += col_width + col_gap
                y = y_top
                ax.text(x + col_width / 2.0, y, group_title, ha="center", va="top", fontsize=title_fs)
                y -= (pad_title + line_gap)

            if color == "BLANK_BOX":
                draw_blank_box(x + indent, y)
            elif color is not None:
                ax.add_patch(Rectangle((x + indent, y - box + 0.06), box, box,
                                       facecolor=color, edgecolor="none"))

            ax.text(x + indent + box + 0.25, y, label, ha="left", va="top", fontsize=item_fs)
            y -= line_gap

        y -= pad_group

def make_global_colors(d, SOFTEN_ALL=True, SOFT_ALPHA=0.01):
    """
    Build stable colors across years.
    - Tissue: Pastel1 (soft qualitative)
    - Geo: Set3 (soft qualitative, many categories)
    - Pest burden: YlOrBr with deeper band
    - Binned numeric rows: BuPu with deeper band
    - BLANK: None (not drawn)
    """
    colors = {}

    def maybe_soften(c):
        if (c is None) or (not SOFTEN_ALL):
            return c
        return blend_with_white(c, alpha=SOFT_ALPHA)

    # -----------------------------
    # Tissue (soft qualitative)
    # -----------------------------
    cmap_tissue = plt.get_cmap("Set2")
    tissues = sorted(d["tissue"].fillna("BLANK").astype(str).unique().tolist()) if "tissue" in d.columns else []
    colors["tissue"] = {t: maybe_soften(cmap_tissue(i % cmap_tissue.N)) for i, t in enumerate(tissues)}
    if "BLANK" in colors["tissue"]:
        colors["tissue"]["BLANK"] = None

    # -----------------------------
    # Geo (soft qualitative with more distinct slots)
    # -----------------------------
    cmap_geo = plt.get_cmap("Set2")
    if "geo_cluster_100km_label" in d.columns:
        geos = sorted(d["geo_cluster_100km_label"].fillna("BLANK").astype(str).unique().tolist())
    else:
        geos = ["BLANK"]
    colors["geo_cluster_100km_label"] = {g: maybe_soften(cmap_geo(i % cmap_geo.N)) for i, g in enumerate(geos)}
    if "BLANK" in colors["geo_cluster_100km_label"]:
        colors["geo_cluster_100km_label"]["BLANK"] = None

    # -----------------------------
    # # Pest burden (bin6 ranges) - deeper sequential band
    # # -----------------------------
    # cmap_pest = plt.get_cmap("YlOrBr")
    # PEST_LABELS_FIXED = ["≤ 1", "1–10", "10–100", "100–1k", "1k–5k", "> 5k"]

    # levels = np.linspace(0.32, 0.88, len(PEST_LABELS_FIXED))

    # pest_map = {"BLANK": None}
    # for lab, lv in zip(PEST_LABELS_FIXED, levels):
    #     pest_map[lab] = maybe_soften(cmap_pest(lv))

    # colors["pest_sum_all_bin6_label"] = pest_map

    # -----------------------------
    # Binned numeric rows - deeper sequential band
    # -----------------------------
    cmap_bins = plt.get_cmap("BuPu")

    bin_levels = np.linspace(0.28, 0.82, len(BIN_LABELS))
    label_to_color = {lab: maybe_soften(cmap_bins(bin_levels[i])) for i, lab in enumerate(BIN_LABELS)}
    label_to_color["BLANK"] = None

    for col, _, _ in BINNED_ROWS:
        colors[col] = dict(label_to_color)

    return colors

In [61]:
# -----------------------------
# TEXT STYLES (edit only these)
# -----------------------------
FONT_FAMILY = "DejaVu Sans"   # e.g., "Arial", "Helvetica", "Times New Roman"
TITLE_FS    = 18
CROP_FS     = 16

LEFT_LABEL_FS   = 12   # left-side row labels (Tissue/Geo/Heat row names)
LEFT_LABEL_WT   = "normal"  # "normal" or "bold"

LEG_TITLE_FS    = 12   # legend titles (Tissue / Geo / Pest burden)
LEG_ITEM_FS     = 11   # legend items (A,G,H / G0.. / 1..6)
LEG_TEXT_WT     = "normal"


In [62]:
# (네 draw_year_diagram은 길어서 “추가/수정해야 하는 부분만” 정확히 콕 집어서 바꿔줄게)
# 아래 3군데만 바꾸면 됨:
#   (1) ANNOT_ROWS에 pathogen_cat5를 pest burden 바로 아래에 넣기
#   (3) 오른쪽 Pathogen legend 좌표 추가 + draw_side_legend_abs 호출

def draw_year_diagram(df_year, year, out_path, global_colors):
    d = df_year.copy()
    d = d[d["crop"].isin(CROPS_KEEP)].copy()

    # -----------------------------
    # Paper-style rendering knobs
    # -----------------------------
    EDGE_COLOR = (0.93, 0.93, 0.93, 1.0)
    EDGE_LW_ANN = 0.40
    EDGE_LW_HEAT = 0.35
    NA_TILE_COLOR = (1.0, 1.0, 1.0, 1.0)

    HEAT_CMAP_NAME = "YlGn_r"

    SOFTEN_HEAT = False
    HEAT_SOFT_ALPHA = 0.10

    def blend_with_white(rgba, alpha=0.10):
        if rgba is None:
            return None
        r, g, b, a = rgba
        r2 = r + (1 - r) * alpha
        g2 = g + (1 - g) * alpha
        b2 = b + (1 - b) * alpha
        return (r2, g2, b2, a)

    # -----------------------------
    # Sort + limit
    # -----------------------------
    keys = [k for k in SORT_KEYS if k in d.columns]
    if keys:
        d = d.sort_values(keys, kind="mergesort")

    if len(d) > MAX_N:
        d = d.iloc[:MAX_N].copy()

    if len(d) == 0:
        print(f"[WARN] No rows for year={year}")
        return

    parts = [d[d["crop"].astype(str) == c] for c in CROPS_ORDER if (d["crop"].astype(str) == c).any()]
    d = pd.concat(parts, axis=0).reset_index(drop=True)
    n = len(d)

    # -----------------------------
    # Heat arrays (continuous; normalized 0..1)
    # -----------------------------
    heat = []
    for col, label in HEAT_ROWS:
        if col in d.columns:
            heat.append((col, label, minmax01(d[col]).to_numpy()))

    # -----------------------------
    # X positions per crop block
    # -----------------------------
    crops = d["crop"].astype(str).fillna("BLANK").to_list()
    x_pos = np.zeros(n, dtype=float)
    cur_x = 0.0
    crop_spans = []

    i = 0
    while i < n:
        crop_i = crops[i]
        j = i
        while j < n and crops[j] == crop_i:
            j += 1

        sx = cur_x
        for t in range(i, j):
            x_pos[t] = cur_x
            cur_x += tile
        ex = cur_x
        crop_spans.append((crop_i, sx, ex))

        if j < n:
            cur_x += GAP_CROP
        i = j

    total_width = cur_x

    # -----------------------------
    # Geometry
    # -----------------------------
    ann_present = [r for r in ANNOT_ROWS if r[0] in d.columns]
    ann_block_h  = len(ann_present) * (tile + ann_row_gap)
    heat_block_h = len(heat) * (tile + heat_row_gap)
    H = top_pad + ann_block_h + gap_ann_to_heat + heat_block_h + 1.9

    x0 = left_pad
    W = x0 + total_width + right_pad

    # -----------------------------
    # FIXED COORDINATES (ADD PATH LEGEND HERE)
    # -----------------------------
    tiles_right = x0 + total_width

    TISSUE_LEG_X = tiles_right + 2.2
    TISSUE_LEG_Y = H - 1.8

    GEO_LEG_X    = tiles_right + 9.5
    GEO_LEG_Y    = H - 1.8

    PEST_LEG_X   = tiles_right + 4
    PEST_LEG_Y   = H * 0.50


    CBAR_X = tiles_right + 14.0
    CBAR_Y = H * 0.1
    CBAR_H = H * 0.4

    W = max(W, CBAR_X + 2.0)

    # -----------------------------
    # Figure / Axes
    # -----------------------------
    fig = plt.figure(figsize=(max(12, W / 7.0), max(6, H / 3.0)))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.axis("off")

    # Title
    ax.text(
        x0 + 50, H - 0.55,
        f"Honey bee dataset (HBB/CRA/CAC/CAS, T2) {year} (n={n})",
        fontsize=TITLE_FS, fontfamily=FONT_FAMILY, va="top", ha="left"
    )

    # Crop headers
    header_y = H - top_pad + 0.30
    for crop, sx, ex in crop_spans:
        ax.text(x0 + (sx + ex) / 2.0, header_y, crop, fontsize=16, ha="center", va="bottom")

    # -----------------------------
    # Legend drawer
    # -----------------------------
    def draw_side_legend_abs(ax, x_left, y_top, title, items,
                             title_fs=12, item_fs=11, line_gap=1, box=1, indent=1):
        ax.text(x_left + 2.8, y_top, title, ha="center", va="top", fontsize=title_fs)
        yy = y_top - (0.25 + line_gap)
        for lab, col in items:
            if col is not None:
                ax.add_patch(Rectangle(
                    (x_left + indent, yy - box + 0.06), box, box,
                    facecolor=col, edgecolor=EDGE_COLOR, linewidth=0.35
                ))
            ax.text(x_left + indent + box + 0.22, yy, str(lab), ha="left", va="top", fontsize=item_fs)
            yy -= line_gap

    # -----------------------------
    # Draw annotations
    # -----------------------------
    y = H - top_pad - 0.9

    for col, label, show_left in ANNOT_ROWS:
        if col not in d.columns:
            continue

        if show_left:
            ax.text(left_pad - 0.7, y + 0.5 * tile, label, fontsize=12, va="center", ha="right")

        cmap = global_colors.get(col, {})
        for idx in range(n):
            key = "BLANK" if pd.isna(d.loc[idx, col]) else str(d.loc[idx, col])
            color = cmap.get(key, None)

            if color is None:
                if DRAW_NA_TILES:
                    color = NA_TILE_COLOR
                else:
                    continue

            ax.add_patch(Rectangle(
                (x0 + x_pos[idx], y),
                tile * 0.92, tile * 0.92,
                facecolor=color, edgecolor=EDGE_COLOR, linewidth=EDGE_LW_ANN
            ))

        y -= (tile + ann_row_gap)

    # Gap
    y -= gap_ann_to_heat

    # -----------------------------
    # Draw heat rows (continuous)
    # -----------------------------
    base = plt.get_cmap(HEAT_CMAP_NAME)
    import matplotlib.colors as mcolors

    
    cmap_heat = mcolors.LinearSegmentedColormap.from_list(
        "trunc",
        base(np.linspace(0.10, 0.90, 256))
    )

    for col, label, vals01 in heat:
        ax.text(left_pad - 0.7, y + 0.5 * tile, label,
                fontsize=12, va="center", ha="right", fontfamily=FONT_FAMILY)

        for idx in range(n):
            v = vals01[idx]
            if pd.isna(v):
                if DRAW_NA_TILES:
                    color = NA_TILE_COLOR
                else:
                    continue
            else:
                # color = cmap_heat(float(np.clip(v, 0.0, 1.0)))
                v = float(np.clip(v, 0.0, 1.0))

                if v == 0.0:
                    color = (1.0, 1.0, 1.0, 1.0)   # 0은 pure white
                else:
                    color = cmap_heat(v)
                if SOFTEN_HEAT:
                    color = blend_with_white(color, alpha=HEAT_SOFT_ALPHA)

            ax.add_patch(Rectangle(
                (x0 + x_pos[idx], y),
                tile * 0.92, tile * 0.92,
                facecolor=color, edgecolor=EDGE_COLOR, linewidth=EDGE_LW_HEAT
            ))
        y -= (tile + heat_row_gap)

    # -----------------------------
    # ABSOLUTE legends (Tissue / Geo / Pest / Pathogen)
    # -----------------------------
    if "tissue" in d.columns:
        m = global_colors.get("tissue", {})
        keys = [k for k in sorted(m.keys()) if k != "BLANK" and m[k] is not None]
        items = [(k, m[k]) for k in keys]
        draw_side_legend_abs(ax, TISSUE_LEG_X, TISSUE_LEG_Y, "Tissue", items)

    if "geo_cluster_100km_label" in d.columns:
        m = global_colors.get("geo_cluster_100km_label", {})
        keys = [k for k in sorted(m.keys()) if k != "BLANK" and m[k] is not None]
        items = [(k, m[k]) for k in keys]
        draw_side_legend_abs(ax, GEO_LEG_X, GEO_LEG_Y, "Geo (100km)", items)

    # if "pest_sum_all_bin6_label" in d.columns:
    #     m = global_colors.get("pest_sum_all_bin6_label", {})
    #     PEST_LABELS = ["≤ 1", "1–10", "10–100", "100–1k", "1k–5k", "> 5k"]
    #     keys = [k for k in PEST_LABELS if (k in m and m[k] is not None)]
    #     items = [(k, m[k]) for k in keys]
    #     draw_side_legend_abs(ax, PEST_LEG_X, PEST_LEG_Y, "Pest burden", items)


    # -----------------------------
    # Colorbar (for heat rows)
    # -----------------------------
    if len(heat) > 0:
        import matplotlib as mpl
        cbar_w = 0.35
        cbar_h = max(3.0, CBAR_H)
        cax = fig.add_axes([CBAR_X / W, CBAR_Y / H, cbar_w / W, cbar_h / H])

        norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
        sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap_heat)
        sm.set_array([])

        cb = fig.colorbar(sm, cax=cax)
        cb.set_label("Heat rows (min-max normalized)", fontsize=11)
        cb.ax.tick_params(labelsize=10)

    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_path)


In [63]:
# ============================================================
# Cell 5) Run 2020 / 2021 + Preview
# ============================================================

# Build global colors from full dataset (stable across years)
global_colors = make_global_colors(df)

draw_year_diagram(df[df["year"] == 2020].copy(), 2020, OUT_2020, global_colors)
draw_year_diagram(df[df["year"] == 2021].copy(), 2021, OUT_2021, global_colors)

Saved: c:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop\diagram_2020.pdf
Saved: c:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop\diagram_2021.pdf


In [64]:
from pathlib import Path
import matplotlib as mpl
mpl.rcParams["svg.fonttype"] = "none"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent   # code 밖으로 한 단계

OUTDIR = PROJECT_ROOT / "figures" / "crop"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_2020 = OUTDIR / "diagram_2020.svg"
OUT_2021 = OUTDIR / "diagram_2021.svg"

print("CWD:", Path.cwd().resolve())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTDIR:", OUTDIR)

global_colors = make_global_colors(df)
draw_year_diagram(df[df["year"] == 2020].copy(), 2020, OUT_2020, global_colors)
draw_year_diagram(df[df["year"] == 2021].copy(), 2021, OUT_2021, global_colors)


CWD: C:\Users\junse\Documents\research\demographics\diagram1_overall\code
PROJECT_ROOT: C:\Users\junse\Documents\research\demographics\diagram1_overall
OUTDIR: C:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop
Saved: C:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop\diagram_2020.svg
Saved: C:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop\diagram_2021.svg


In [65]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTDIR = PROJECT_ROOT / "figures" / "crop"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_2020 = OUTDIR / "diagram_2020.png"
OUT_2021 = OUTDIR / "diagram_2021.png"

print("CWD:", Path.cwd().resolve())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTDIR:", OUTDIR)

global_colors = make_global_colors(df)
draw_year_diagram(df[df["year"] == 2020].copy(), 2020, OUT_2020, global_colors)
draw_year_diagram(df[df["year"] == 2021].copy(), 2021, OUT_2021, global_colors)


CWD: C:\Users\junse\Documents\research\demographics\diagram1_overall\code
PROJECT_ROOT: C:\Users\junse\Documents\research\demographics\diagram1_overall
OUTDIR: C:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop
Saved: C:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop\diagram_2020.png
Saved: C:\Users\junse\Documents\research\demographics\diagram1_overall\figures\crop\diagram_2021.png
